In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../'))
sys.path.append(base_path)

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

import numpy as np
from tqdm import tqdm
import datetime

from data_loader_complete import DataLoader
import project_images_loader as pil

import discriminator as Discriminator
import generator as Generator
import feature_extractor as feature_extractor

import utils.functions as fc
import utils.util_deep_learning as udl
import utils.util as util

tf.keras.backend.set_image_data_format("channels_first")



2024-03-22 17:22:25.242174: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-22 17:22:25.242244: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-22 17:22:25.244055: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-22 17:22:25.255158: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-22 17:22:26.615172: W tensorflow/compiler/tf2

## Visualize data

In [2]:
# %matplotlib widget

# data_train = DataLoader(batch_size=3, image_size=(193,229,193), shuffle=False, channels_first=True, image_sufix=pil.INAME_PREPROCESSED, nb_patients=-1)
# fc.imgshow_3D("", data_train[0][0][0,0])

### Loss functions

In [3]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = (real_loss + fake_loss)/2
    return total_loss

def generator_loss(fake_output, y_batch, y_batch_fake, feature_y_list, feature_fake_y_list):
    gan_loss = cross_entropy(tf.ones_like(fake_output), fake_output)

    l1_loss = tf.reduce_mean(tf.abs(y_batch - y_batch_fake))
    percetual_loss = tf.reduce_mean(tf.abs(feature_y_list - feature_fake_y_list))

    total_loss = gan_loss + 25.0 * l1_loss + 25.0 * percetual_loss

    return total_loss, gan_loss, l1_loss, percetual_loss


In [4]:
# @tf.function
# def val_step(model, x_images, y_true, masks, class_centers, epoch):
#     y_pred, x_feat, x_gap = model(x_images, training=False)
#     x_weights = model.layers[-1].weights[0]
#     loss, loss_ce, loss_cskl, loss_cl, loss_cskl_mask = combined_loss(y_true, y_pred, x_feat, x_weights, masks, x_gap, class_centers, epoch)
#     return y_pred, loss, loss_ce, loss_cskl, loss_cl, loss_cskl_mask

@tf.function
def train_step(G, D, Fe, x_batch, y_batch, g_optimizer, d_optimizer):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        y_batch_fake = G(x_batch, training=True)

        feature_y_list_fake = Fe(y_batch_fake, training=False)
        feature_y_list = Fe(y_batch, training=False)

        disc_output_real = D([x_batch, y_batch], training=True)
        disc_output_fake = D([x_batch, y_batch_fake], training=True)

        disc_loss = discriminator_loss(disc_output_real, disc_output_fake)
        gen_loss, gan_loss, l1_loss, percetual_loss = generator_loss(disc_output_fake, y_batch, y_batch_fake, feature_y_list, feature_y_list_fake)

    gradients_of_generator = gen_tape.gradient(gen_loss, G.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, D.trainable_variables)

    g_optimizer.apply_gradients(zip(gradients_of_generator, G.trainable_variables))
    d_optimizer.apply_gradients(zip(gradients_of_discriminator, D.trainable_variables))

    return gen_loss, disc_loss, gan_loss, l1_loss, percetual_loss



### Training

In [5]:

def train(num_epochs, batch_size):
    # input_shape = (193,229,193)
    input_shape = (192,224,192)

    # ---- Models
    D = Discriminator.create_discriminator_model(input_shape)
    G = Generator.create_generator_model(input_shape)
    Fe = feature_extractor.create_features_extractor((1,)+input_shape, f"{current_path}/models/genesis_model/Genesis_Chest_CT.h5")
    
    # Freeze features extractor layers
    Fe.trainable = False
    udl.freeze_layers(Fe, 0)

    # optimizers
    inital_lr = 1e-4
    g_optimizer = tf.keras.optimizers.Adam(inital_lr)
    d_optimizer = tf.keras.optimizers.Adam(inital_lr)

    # ---- Data
    data_train = DataLoader(batch_size=batch_size, image_size=input_shape, shuffle=True, channels_first=True, image_sufix=pil.INAME_PREPROCESSED, nb_patients=300, shuffle_before_selecting=False)
    # data_val = DataLoader(f"{pil.get_base_folder()}data_test/val", batch_size=batch_size, image_size=input_shape, channels_first=False)

    # visualization
    nb_loss = 5
    acum_loss_train = udl.AcumulativeMetrics(nb_metrics = nb_loss)
    # tensorboard_callback = TensorBoard(log_dir=f"{current_path}/models/logs")

    log_dir=f"{current_path}/models/logs/"
    summary_writer = tf.summary.create_file_writer(
    log_dir + "fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    # %load_ext tensorboard
    # %tensorboard --logdir {log_dir}

    # ---- checkpoint
    checkpoint_dir = f"{current_path}/models/training_checkpoints/"
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
    checkpoint = tf.train.Checkpoint(generator_optimizer=g_optimizer,
                                    discriminator_optimizer=d_optimizer,
                                    generator=G,
                                    discriminator=D)
    # checkpoint_manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir)
    

    for epoch in range(0,num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # ---- Train
        acum_loss_train.reset()
        progress_bar = tqdm(total=len(data_train), desc=f'Initializing...')
        for step, (x_batch, y_batch) in enumerate(data_train):
            gen_loss, disc_loss, gan_loss, l1_loss, percetual_loss = train_step(G, D, Fe, x_batch, y_batch, g_optimizer, d_optimizer)
            acum_loss_train.update((gen_loss, disc_loss, gan_loss, l1_loss, percetual_loss))
            loss_mean = acum_loss_train.get_mean()
            
            # img_sim = util.compute_ssim(y_batch[0,0], y_batch_fake[0,0])
            img_sim = 0.0
            progress_bar.set_description(desc=f'Sim: {img_sim:.5f} \t gen_loss: {loss_mean[0]:.5f} \t disc_loss: {loss_mean[1]:.5f} \t gan_loss: {loss_mean[2]:.5f} \t l1_loss: {loss_mean[3]:.5f}\t p_loss: {loss_mean[4]:.5f}')
            progress_bar.update(1)
        del progress_bar

        # ---- Check points
        if  epoch > 0 and epoch % 2 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
        
        # ---- Learning rate
        # if epoch > 44 and epoch % 22 == 0:
        if epoch > 44 and epoch % 22 == 0:
            inital_lr *= 0.5
            g_optimizer.learning_rate = inital_lr
            d_optimizer.learning_rate = inital_lr

        # ---- tensorboard
        loss_mean = acum_loss_train.get_mean()
        with summary_writer.as_default():
            tf.summary.scalar('generator_loss', loss_mean[0], step=epoch)
            tf.summary.scalar('discriminator_loss', loss_mean[1], step=epoch)
            tf.summary.scalar('gan_loss', loss_mean[2], step=epoch)
            tf.summary.scalar('l1_loss', loss_mean[3], step=epoch)
            tf.summary.scalar('p_loss', loss_mean[4], step=epoch)
        
            

train(num_epochs=50, batch_size=1)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


2024-03-22 17:22:48.870806: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38374 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:82:00.0, compute capability: 8.0



Epoch 1/50


Initializing...:   0%|          | 0/300 [00:00<?, ?it/s]"`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
2024-03-22 17:23:09.608340: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-03-22 17:23:12.281732: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-03-22 17:23:17.743494: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


In [ ]:
# Sim: 0.00000 	 gen_loss: 0.93415 	 disc_loss: 1.11709: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]
# Sim: 0.00000 	 gen_loss: 1.22538 	 disc_loss: 1.64933: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]